# Lab 11: Keypoints, SIFT and RANSAC

*Written by Sai Raj Kishore Perla, modified by Yalda.*

Two photos of the same place: which pixel in one photo is which pixel in the other? If we can answer that for a few hundred
points, we can find the photo inside another photo, stitch a panorama, or (later) recover 3D.

The recipe has three steps, and we build each one ourselves before using OpenCV's version:

1. **Corners (Harris).** Points that look different from all their neighbours, so we can find them again.
2. **Scale (Difference of Gaussians).** Find each point's size too, so zooming in does not matter.
3. **Descriptors and matching (SIFT).** Describe each point with 128 numbers and find its twin in the other photo.
4. **RANSAC.** Some matches are wrong. Fit a model that ignores them.
5. **Homographies.** The model that maps one photo onto the other. Find a zoomed photo inside the wide photo.
6. **Panorama.** Stitch two views into one picture.
7. **Try it yourself.** Your own two photos.

**Extra** (in class if you have time, otherwise at home): ORB, the fast binary alternative to SIFT.

| Part | Time |
|---|---|
| Setup + 1. Corners | 30 min |
| 2. Scale | 20 min |
| 3. Descriptors and matching | 25 min |
| 4. RANSAC | 25 min |
| 5. Homographies | 15 min |
| 6. Panorama | 15 min |
| 7. Try it yourself | 10 min |
| Extra. ORB | 15 min |

For each **TODO**, write your code, then run the check cell below it. If you are stuck, open the hidden **Solution** cell.

## Environment (skip this on Colab)

Colab has everything already. To run this notebook on your own machine or a lab machine, create a conda environment
**once** from a terminal (or use the `vc` environment from the earlier labs):

```bash
conda create -n vc python=3.11 -y && conda activate vc && pip install numpy matplotlib opencv-python jupyter ipykernel && python -m ipykernel install --user --name vc
```

Then pick the **vc** kernel in Jupyter / VS Code (top right). No GPU is needed today.

If you already have an environment, the next cell installs anything missing into the kernel you are using.
If it installs something, **restart the kernel** (*Kernel → Restart*), then continue.

In [ ]:
import importlib.util
if importlib.util.find_spec("cv2") is None:          # install OpenCV only if it is missing
    %pip install -q opencv-python
else:
    print("OpenCV is already installed")

In [ ]:
# check your environment
import sys
print("python:", sys.version.split()[0], " from", sys.executable)
try:
    import cv2
    import numpy
    import matplotlib
    print("opencv:", cv2.__version__, " numpy:", numpy.__version__, " matplotlib:", matplotlib.__version__)
    print("SIFT available:", hasattr(cv2, "SIFT_create"))
except ImportError as e:
    print("ERROR:", e)
    print("Restart the kernel and run this cell again, or pick the vc kernel (see above).")

## Setup

Two photos of the SFU Burnaby courtyard (Academic Quadrangle): a wide one, and a zoomed one taken closer to the rock.

In [ ]:
import os
import math
import time
import urllib.request

import numpy as np
import cv2
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 100
np.set_printoptions(suppress=True, precision=4)

URL = "https://raw.githubusercontent.com/arash-mham/visual-computing-I/main/lab-material/lab11/"


def download(name):
    if not os.path.exists(name):
        urllib.request.urlretrieve(URL + name, name)


def load(name):
    # an RGB image (uint8)
    return cv2.cvtColor(cv2.imread(name), cv2.COLOR_BGR2RGB)


def to_gray(img):
    # a gray image with values 0..1 (float32)
    return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255


def show_matches(img1, img2, pts1, pts2, colors="lime", title="", lw=0.8):
    # img1 and img2 side by side, a line from pts1[i] (in img1) to pts2[i] (in img2)
    h = max(img1.shape[0], img2.shape[0])
    canvas = np.zeros((h, img1.shape[1] + img2.shape[1], 3), np.uint8)
    canvas[:img1.shape[0], :img1.shape[1]] = img1
    canvas[:img2.shape[0], img1.shape[1]:] = img2
    if isinstance(colors, str):
        colors = [colors] * len(pts1)
    plt.figure(figsize=(14, 14 * h / canvas.shape[1]))
    plt.imshow(canvas)
    for (x1, y1), (x2, y2), c in zip(pts1, pts2, colors):
        plt.plot([x1, x2 + img1.shape[1]], [y1, y2], color=c, lw=lw)
        plt.plot([x1, x2 + img1.shape[1]], [y1, y2], ".", color=c, ms=3)
    plt.title(title)
    plt.axis("off")
    plt.show()


download("courtyard_wide.jpg")
download("courtyard_zoom.jpg")
wide = load("courtyard_wide.jpg")
zoom = load("courtyard_zoom.jpg")
print("wide:", wide.shape, " zoom:", zoom.shape)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
ax[0].imshow(wide)
ax[0].set_title("wide")
ax[1].imshow(zoom)
ax[1].set_title("zoom")
for a in ax:
    a.axis("off")
plt.show()

## Part 1: Corners (Harris)

A good keypoint is a point we can find again. Look at a small window around a point and move the window a little:

- **flat** region (sky): nothing changes, whichever way we move,
- **edge** (roof line): nothing changes when we move along the edge,
- **corner**: everything changes, whichever way we move. That is the point we want.

The image gradients $I_x$, $I_y$ inside the window tell us which case we are in. Each dot below is one pixel of the
window, placed at $(I_x, I_y)$.

In [ ]:
gray = to_gray(wide)
Ix = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
Iy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

points = {"flat": (150, 50), "edge": (296, 131), "corner": (99, 286)}   # (x, y) in the wide photo
r = 10                                                                  # window = 21 x 21 pixels

fig, ax = plt.subplot_mosaic([["photo", "flat", "edge", "corner"], ["photo", "s_flat", "s_edge", "s_corner"]],
                             figsize=(16, 6.5), gridspec_kw={"width_ratios": [2.2, 1, 1, 1]})
ax["photo"].imshow(wide)
for name, (x, y) in points.items():
    ax["photo"].add_patch(plt.Rectangle((x - r, y - r), 2 * r, 2 * r, fill=False, color="yellow", lw=2))
    ax["photo"].text(x + r + 4, y, name, color="yellow", fontsize=11, va="center")
ax["photo"].axis("off")
for name, (x, y) in points.items():
    ax[name].imshow(wide[y - r:y + r + 1, x - r:x + r + 1])
    ax[name].set_title(name + " window")
    ax[name].axis("off")
    gx = Ix[y - r:y + r + 1, x - r:x + r + 1].ravel()
    gy = Iy[y - r:y + r + 1, x - r:x + r + 1].ravel()
    a = ax["s_" + name]
    a.scatter(gx, gy, s=6)
    a.set_xlim(-2, 2)
    a.set_ylim(-2, 2)
    a.set_aspect("equal")
    a.axhline(0, color="gray", lw=0.5)
    a.axvline(0, color="gray", lw=0.5)
    a.set_xlabel("$I_x$")
    a.set_ylabel("$I_y$")
plt.tight_layout()
plt.show()

Flat: all dots near the centre. Edge: dots along **one** line. Corner: dots spread in **two** directions.

The **structure tensor** summarizes that cloud of dots with a $2\times2$ matrix (a weighted sum over the window, with
Gaussian weights $w$):

$$M = \sum_{\text{window}} w \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$$

Its two eigenvalues $\lambda_1, \lambda_2$ are the spread of the cloud in its two main directions: both small = flat,
one large = edge, both large = corner. Harris avoids computing eigenvalues with one score:

$$R = \det M - k\,(\operatorname{trace} M)^2 = \lambda_1\lambda_2 - k\,(\lambda_1+\lambda_2)^2, \qquad k \approx 0.05$$

$R > 0$ at corners, $R < 0$ at edges, $R \approx 0$ in flat regions.

### TODO 1: `harris_response`

The weighted sum over the window is a Gaussian blur of each product image: `cv2.GaussianBlur(img, (0, 0), sigma)`.

In [ ]:
def harris_response(gray, sigma=2.0, k=0.05):
    # gray: 2D float32 image. Returns R, an image of the same shape.
    # TODO: gradients Ix, Iy with cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3) and (..., 0, 1, ...)
    # TODO: the three entries of M: blur Ix*Ix, Iy*Iy and Ix*Iy with cv2.GaussianBlur(..., (0, 0), sigma)
    # TODO: R = det(M) - k * trace(M)^2
    R = ...
    return R

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def harris_response(gray, sigma=2.0, k=0.05):
    Ix = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    Iy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    Sxx = cv2.GaussianBlur(Ix * Ix, (0, 0), sigma)
    Syy = cv2.GaussianBlur(Iy * Iy, (0, 0), sigma)
    Sxy = cv2.GaussianBlur(Ix * Iy, (0, 0), sigma)
    det = Sxx * Syy - Sxy * Sxy
    trace = Sxx + Syy
    R = det - k * trace ** 2
    return R

In [ ]:
# run this cell to check your code
square = np.zeros((100, 100), np.float32)
square[30:70, 30:70] = 1
R_sq = harris_response(square)
ok_shape = R_sq.shape == square.shape
ok_corner = R_sq[27:33, 27:33].max() > 1
ok_edge = R_sq[50, 27:33].min() < -0.5
ok_flat = abs(R_sq[50, 50]) < 1e-4 and abs(R_sq[5, 5]) < 1e-4
print("shape:", ok_shape, " corner R > 0:", ok_corner, " edge R < 0:", ok_edge, " flat R ~ 0:", ok_flat)
print("correct:", bool(ok_shape and ok_corner and ok_edge and ok_flat))

$R$ on the photo (red = corner, blue = edge), and the 300 strongest corners. A corner is kept only if it is the largest
$R$ in its $15\times15$ neighbourhood (non-maximum suppression), otherwise we would get a clump of points per corner.

In [ ]:
def find_peaks(R, n, radius=7):
    # the n largest local maxima of R (x and y arrays)
    local_max = R == cv2.dilate(R, np.ones((2 * radius + 1, 2 * radius + 1), np.uint8))
    ys, xs = np.nonzero(local_max & (R > 0))
    order = np.argsort(-R[ys, xs])[:n]
    return xs[order], ys[order]


R = harris_response(to_gray(wide))
xs, ys = find_peaks(R, 300)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
v = np.percentile(np.abs(R), 99.5)
ax[0].imshow(R, cmap="bwr", vmin=-v, vmax=v)
ax[0].set_title("Harris R  (red: corner, blue: edge)")
ax[1].imshow(wide)
ax[1].plot(xs, ys, "o", ms=4, mfc="none", mec="red", mew=1.2)
ax[1].set_title("300 strongest corners")
for a in ax:
    a.axis("off")
plt.show()

**One problem: size.** Harris always looks through the same 21×21 window. Below is the same rounded corner at two
sizes. In the small image the window holds the whole corner, so $R > 0$. Zoomed in 4 times, the window holds only a
piece of the curve, which looks like an edge, so $R < 0$. The corner is lost:

In [ ]:
def rounded_corner(size, start, radius):
    # a white square with a rounded top-left corner, and the point in the middle of the curve
    img = np.zeros((size, size), np.float32)
    c = start + radius
    cv2.rectangle(img, (c, start), (size, size), 1.0, -1)
    cv2.rectangle(img, (start, c), (size, size), 1.0, -1)
    cv2.circle(img, (c, c), radius, 1.0, -1, lineType=cv2.LINE_AA)
    p = round(c - radius / np.sqrt(2))
    return img, p


small, p1 = rounded_corner(60, 20, 5)        # a slightly rounded corner
big, p2 = rounded_corner(240, 80, 20)        # the same shape, 4 times bigger
r = 10                                       # the same 21 x 21 window in both

fig, ax = plt.subplots(1, 4, figsize=(16, 4.4))
for i, (img, p, name) in enumerate([(small, p1, "small"), (big, p2, "zoomed in 4x")]):
    R_img = harris_response(img)
    R_here = R_img[p - 2:p + 3, p - 2:p + 3].max()
    ax[2 * i].imshow(img, cmap="gray")
    ax[2 * i].add_patch(plt.Rectangle((p - r, p - r), 2 * r, 2 * r, fill=False, color="yellow", lw=2))
    ax[2 * i].set_title(name + ", window in yellow")
    ax[2 * i + 1].imshow(img[p - r:p + r + 1, p - r:p + r + 1], cmap="gray", vmin=0, vmax=1)
    ax[2 * i + 1].set_title(f"inside the window: R = {R_here:.3f}\n" + ("a corner" if R_here > 0 else "an edge"))
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## Part 2: Scale (Difference of Gaussians)

Harris needs the right window size. SIFT finds the size of each keypoint like this:

1. Blur the image more and more (a growing $\sigma$).
2. Subtract each blur from the next one: the **Difference of Gaussians** (DoG), $D_\sigma = G_{k\sigma} * I - G_\sigma * I$.
3. A round spot (a blob) gives its strongest DoG at one $\sigma$: when the blur is about as big as the spot
   (radius $\approx \sqrt2\,\sigma$).
4. That $\sigma$ is the keypoint's size.

Test image: 5 white disks with radius 3, 6, 12, 20 and 32 pixels. Each one should get its own best $\sigma$.

In [ ]:
radii = [3, 6, 12, 20, 32]
blobs = np.zeros((140, 400), np.float32)
centers = []
x = 15
for r in radii:
    x = x + r
    cv2.circle(blobs, (x, 70), r, 1.0, -1, lineType=cv2.LINE_AA)
    centers.append(x)
    x = x + r + 25

sigmas = [2 ** (i / 4) for i in range(25)]          # 1 ... 64, each one 2^(1/4) times the previous
print("sigmas:", np.round(sigmas, 1))
plt.figure(figsize=(10, 3.5))
plt.imshow(blobs, cmap="gray")
plt.axis("off")
plt.show()

### TODO 2: `dog_stack`

In [ ]:
def dog_stack(gray, sigmas):
    # Returns a list with len(sigmas) - 1 images: dog[i] = blur(gray, sigmas[i+1]) - blur(gray, sigmas[i])
    # TODO: blur the image with each sigma: cv2.GaussianBlur(gray, (0, 0), s)
    # TODO: subtract neighbouring blurs
    dog = ...
    return dog

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def dog_stack(gray, sigmas):
    blurred = []
    for s in sigmas:
        blurred.append(cv2.GaussianBlur(gray, (0, 0), s))
    dog = []
    for i in range(len(sigmas) - 1):
        dog.append(blurred[i + 1] - blurred[i])
    return dog

In [ ]:
# run this cell to check your code
dog = dog_stack(blobs, sigmas)
ok_len = len(dog) == len(sigmas) - 1
found = []
for c in centers:
    response = [abs(d[70, c]) for d in dog]
    i = int(np.argmax(response))
    sigma = math.sqrt(sigmas[i] * sigmas[i + 1])      # the sigma "between" the two blurs
    found.append(math.sqrt(2) * sigma)
print("true radius: ", radii)
print("found radius:", np.round(found, 1))
ok_r = all(abs(f - r) / r < 0.2 for f, r in zip(found, radii))
print("correct:", bool(ok_len and ok_r))

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(15, 4.5))
for col, i in enumerate([0, 8, 14, 19]):
    ax[0, col].imshow(cv2.GaussianBlur(blobs, (0, 0), sigmas[i]), cmap="gray")
    ax[0, col].set_title(f"blur, sigma = {sigmas[i]:.1f}")
    ax[1, col].imshow(np.abs(dog[i]), cmap="magma")
    ax[1, col].set_title(f"|DoG|, sigma = {sigmas[i]:.1f}")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={"width_ratios": [1, 1.4]})
mid = [math.sqrt(sigmas[i] * sigmas[i + 1]) for i in range(len(dog))]
for r, c in zip(radii, centers):
    ax[0].plot(mid, [abs(d[70, c]) for d in dog], label=f"radius {r}")
ax[0].set_xscale("log")
ax[0].set_xlabel("sigma")
ax[0].set_ylabel("|DoG| at the disk centre")
ax[0].legend()
ax[0].set_title("each disk has its own best sigma")
ax[1].imshow(blobs, cmap="gray")
for f, c in zip(found, centers):
    ax[1].add_patch(plt.Circle((c, 70), f, fill=False, color="red", lw=2))
ax[1].set_title("circles: radius = sqrt(2) * best sigma")
ax[1].axis("off")
plt.show()

### SIFT keypoints

OpenCV's SIFT does this properly: a DoG stack at several image resolutions, the local maxima in $(x, y, \sigma)$,
points on edges removed (with a Harris-like test), and an **orientation** for each point (the main gradient direction
around it). Each keypoint has a position, a size and an angle, drawn below as a circle with a line.

In [ ]:
sift = cv2.SIFT_create(nfeatures=2000)


def sift_features(img):
    gray8 = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    keypoints, descriptors = sift.detectAndCompute(gray8, None)
    return keypoints, descriptors


def draw_keypoints(ax, keypoints, color="yellow", x0=0, y0=0):
    # (x0, y0): top-left corner of the shown crop
    for kp in keypoints:
        x = kp.pt[0] - x0
        y = kp.pt[1] - y0
        r = kp.size / 2
        a = np.deg2rad(kp.angle)
        ax.add_patch(plt.Circle((x, y), r, fill=False, color=color, lw=1))
        ax.plot([x, x + r * np.cos(a)], [y, y + r * np.sin(a)], color=color, lw=1)


kp_wide, desc_wide = sift_features(wide)
print("keypoints:", len(kp_wide), " descriptors:", desc_wide.shape)
big_ones = [kp for kp in kp_wide if kp.size > 10]

fig, ax = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1.5, 1]})
ax[0].imshow(wide)
draw_keypoints(ax[0], big_ones)
ax[0].set_title(f"the {len(big_ones)} keypoints larger than 10 pixels (of {len(kp_wide)})")
ax[1].imshow(wide[230:470, 150:450])
draw_keypoints(ax[1], [kp for kp in kp_wide if 150 < kp.pt[0] < 450 and 230 < kp.pt[1] < 470 and kp.size > 6], x0=150, y0=230)
ax[1].set_xlim(0, 300)
ax[1].set_ylim(240, 0)
ax[1].set_title("zoom: keypoints larger than 6 pixels")
for a in ax:
    a.axis("off")
plt.show()

## Part 3: Descriptors and matching

To match a keypoint we need a **descriptor**: 128 numbers that describe what is inside its circle.

1. Take the patch inside the circle, resized to the keypoint's size and turned by its angle.
2. Split it into $4\times4$ cells.
3. In each cell, count the gradient directions in 8 bins (a histogram).

$4\times4\times8 = 128$ numbers. Because the patch follows the keypoint's size and angle, the same point gets almost the
same 128 numbers in a zoomed or rotated photo.

To test this, we make a second photo ourselves: the wide photo **rotated 30°, shrunk to 0.7× and darker** (to test
lighting too). Since we made it, we know where every point went.

In [ ]:
M = cv2.getRotationMatrix2D((480, 318), 30, 0.7)          # 2x3 matrix: rotate 30 degrees, scale 0.7
rotated = cv2.warpAffine(wide, M, (960, 636))
rotated = np.clip(rotated * 0.8 + 20, 0, 255).astype(np.uint8)
kp_rot, desc_rot = sift_features(rotated)


def true_position(pts):
    # where points of the wide photo land in the rotated photo
    return pts @ M[:, :2].T + M[:, 2]


fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
ax[0].imshow(wide)
ax[0].set_title("wide")
ax[1].imshow(rotated)
ax[1].set_title("rotated 30°, scale 0.7, darker")
for a in ax:
    a.axis("off")
plt.show()

**Matching:** for each descriptor in photo 1, find the closest descriptor in photo 2 (smallest Euclidean distance).

Problem: every point gets a closest one, even points that are not in photo 2 at all. And one window of the building
looks like all the other windows. Lowe's **ratio test** keeps a match only if the closest is clearly closer than the
second closest:

$$\frac{d_{\text{1st}}}{d_{\text{2nd}}} < 0.75$$

### TODO 3: `match_ratio`

In [ ]:
def match_ratio(desc1, desc2, ratio=0.75):
    # desc1: (N1, 128), desc2: (N2, 128). Returns a list of pairs (i, j): desc1[i] matches desc2[j].
    pairs = []
    for i in range(len(desc1)):
        # TODO: distances from desc1[i] to every row of desc2 (np.linalg.norm with axis=1)
        # TODO: the nearest and the second nearest (np.argsort)
        # TODO: keep (i, j_nearest) if nearest distance < ratio * second nearest distance
        ...
    return pairs

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def match_ratio(desc1, desc2, ratio=0.75):
    pairs = []
    for i in range(len(desc1)):
        d = np.linalg.norm(desc2 - desc1[i], axis=1)
        order = np.argsort(d)
        if d[order[0]] < ratio * d[order[1]]:
            pairs.append((i, order[0]))
    return pairs

In [ ]:
# run this cell to check your code
pairs = match_ratio(desc_wide, desc_rot)
knn = cv2.BFMatcher(cv2.NORM_L2).knnMatch(desc_wide, desc_rot, k=2)
pairs_cv = set((m.queryIdx, m.trainIdx) for m, n in knn if m.distance < 0.75 * n.distance)
same = len(set((int(i), int(j)) for i, j in pairs) & pairs_cv)
print("your matches:", len(pairs), " OpenCV's:", len(pairs_cv), " in both:", same)
print("correct:", len(pairs) > 0 and same >= 0.99 * len(pairs_cv) and len(pairs) <= 1.01 * len(pairs_cv))

How good is the ratio test? We know where each point really went, so a match is **correct** if it lands within 3 pixels
of the true position. The histogram shows the ratio of every closest match: correct ones (green) have a small ratio,
wrong ones (red) are close to 1.

In [ ]:
ratios = []
nearest = []
for i in range(len(desc_wide)):
    d = np.linalg.norm(desc_rot - desc_wide[i], axis=1)
    order = np.argsort(d)
    nearest.append(order[0])
    ratios.append(d[order[0]] / d[order[1]])
ratios = np.array(ratios)
p1 = np.array([kp.pt for kp in kp_wide])
p2 = np.array([kp_rot[j].pt for j in nearest])
correct = np.linalg.norm(true_position(p1) - p2, axis=1) < 3

kept = ratios < 0.75
print(f"nearest neighbour only: {len(ratios)} matches, {100 * correct.mean():.0f}% correct")
print(f"with the ratio test:    {kept.sum()} matches, {100 * correct[kept].mean():.0f}% correct")

plt.figure(figsize=(8, 3.5))
bins = np.linspace(0, 1, 41)
plt.hist(ratios[correct], bins, alpha=0.7, color="tab:green", label="correct")
plt.hist(ratios[~correct], bins, alpha=0.7, color="tab:red", label="wrong")
plt.axvline(0.75, color="black", ls="--", label="0.75")
plt.xlabel("d_1st / d_2nd")
plt.ylabel("matches")
plt.legend()
plt.show()

idx = np.nonzero(kept)[0][::8]
show_matches(wide, rotated, p1[idx], p2[idx], ["lime" if correct[i] else "red" for i in idx],
             "ratio test matches (every 8th), green = correct, red = wrong")

Now a real pair: the zoomed photo was taken closer, so things are about 2.5 times bigger.

In [ ]:
kp_zoom, desc_zoom = sift_features(zoom)
pairs_zw = match_ratio(desc_zoom, desc_wide)
pts_zoom = np.array([kp_zoom[i].pt for i, j in pairs_zw])
pts_wide = np.array([kp_wide[j].pt for i, j in pairs_zw])
print("matches:", len(pairs_zw))
show_matches(zoom, wide, pts_zoom, pts_wide, "yellow", "zoom -> wide, ratio test matches", lw=0.6)

Most lines go from the rock to the rock, but some are wrong (water to building, one window to another). Here we do not
know the true positions, so how do we find the wrong ones? Part 4.

## Part 4: RANSAC

A toy problem first: fit a line to points when 40% of them are junk. Least squares (`np.polyfit`) tries to please every
point, so the junk pulls the line away:

In [ ]:
rng = np.random.default_rng(1)
n_in, n_out = 60, 40
x_in = rng.uniform(0, 10, n_in)
y_in = 0.5 * x_in + 2 + rng.normal(0, 0.2, n_in)        # the true line: y = 0.5 x + 2
x_out = rng.uniform(0, 10, n_out)
y_out = rng.uniform(0, 10, n_out)                        # junk
x = np.concatenate([x_in, x_out])
y = np.concatenate([y_in, y_out])
is_inlier = np.arange(len(x)) < n_in                     # only for checking, RANSAC does not see this

a_ls, b_ls = np.polyfit(x, y, 1)
xx = np.array([0, 10])
plt.figure(figsize=(6, 4.5))
plt.scatter(x, y, s=12, color="gray")
plt.plot(xx, 0.5 * xx + 2, "g--", label="true line")
plt.plot(xx, a_ls * xx + b_ls, "r", label=f"least squares: y = {a_ls:.2f} x + {b_ls:.2f}")
plt.legend()
plt.show()

**RANSAC** (RANdom SAmple Consensus) lets the points vote:

1. Pick the smallest random sample that defines a model: **2 points** for a line.
2. Fit the model to them.
3. Count the **inliers**: points closer than a threshold to the model.
4. Repeat many times and keep the model with the most inliers.
5. Fit once more, with least squares, on the inliers only.

A sample with a junk point gives a line few points agree with. One sample of 2 good points is enough to win.

### TODO 4: `ransac_line`

In [ ]:
def ransac_line(x, y, n_iters=100, thresh=0.5):
    # Returns a boolean array: True for the inliers of the best line
    rng = np.random.default_rng(0)
    best_inliers = np.zeros(len(x), dtype=bool)
    for it in range(n_iters):
        # TODO: pick 2 different points: i, j = rng.choice(len(x), 2, replace=False)
        # TODO: the line y = a * x + b through them (skip this sample if x[i] == x[j])
        # TODO: inliers = points with |y - (a * x + b)| < thresh
        # TODO: if there are more inliers than in best_inliers, keep them
        ...
    return best_inliers

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def ransac_line(x, y, n_iters=100, thresh=0.5):
    rng = np.random.default_rng(0)
    best_inliers = np.zeros(len(x), dtype=bool)
    for it in range(n_iters):
        i, j = rng.choice(len(x), 2, replace=False)
        if x[i] == x[j]:
            continue
        a = (y[j] - y[i]) / (x[j] - x[i])
        b = y[i] - a * x[i]
        inliers = np.abs(y - (a * x + b)) < thresh
        if inliers.sum() > best_inliers.sum():
            best_inliers = inliers
    return best_inliers

In [ ]:
# run this cell to check your code
inl = ransac_line(x, y)
a_r, b_r = np.polyfit(x[inl], y[inl], 1)
found_true = (inl & is_inlier).sum()
found_junk = (inl & ~is_inlier).sum()
print(f"inliers found: {found_true} of {n_in} true points, {found_junk} junk points")
print(f"line: y = {a_r:.2f} x + {b_r:.2f}   (true: y = 0.50 x + 2.00)")
print("correct:", bool(found_true >= 0.9 * n_in and found_junk <= 5 and abs(a_r - 0.5) < 0.05 and abs(b_r - 2) < 0.2))

Six RANSAC samples (the two picked points are the big dots, the band is the threshold), and the final fit:

In [ ]:
rng_demo = np.random.default_rng(3)
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
ax = ax.ravel()
best = 0
for k in range(6):
    i, j = rng_demo.choice(len(x), 2, replace=False)
    a = (y[j] - y[i]) / (x[j] - x[i])
    b = y[i] - a * x[i]
    inliers = np.abs(y - (a * x + b)) < 0.5
    best = max(best, inliers.sum())
    ax[k].fill_between(xx, a * xx + b - 0.5, a * xx + b + 0.5, color="tab:blue", alpha=0.2)
    ax[k].scatter(x, y, s=10, c=np.where(inliers, "tab:blue", "lightgray"))
    ax[k].scatter(x[[i, j]], y[[i, j]], s=80, color="black")
    ax[k].set_title(f"sample {k + 1}: {inliers.sum()} inliers (best so far {best})")
    ax[k].set_xlim(0, 10)
    ax[k].set_ylim(-1, 11)
ax[6].scatter(x, y, s=10, c=np.where(inl, "tab:blue", "lightgray"))
ax[6].plot(xx, a_r * xx + b_r, "b", label="RANSAC + refit")
ax[6].plot(xx, a_ls * xx + b_ls, "r", label="least squares")
ax[6].legend()
ax[6].set_title(f"final: {inl.sum()} inliers")
ax[6].set_xlim(0, 10)
ax[6].set_ylim(-1, 11)
ax[7].axis("off")
plt.tight_layout()
plt.show()

**How many tries?** We need at least one try where both picked points are good. Our numbers:

- 60% of the points are good, so one try picks 2 good points with chance $0.6 \times 0.6 = 0.36$.
- So one try fails with chance $1 - 0.36 = 0.64$.
- 11 tries **all** fail with chance $0.64^{11} \approx 0.007$, less than 1%. So 11 tries are enough (we used 100).

The same in general, with $w$ = fraction of good points, $s$ = points per try and $p$ = the success we want (0.99):

$$N = \frac{\log(1-p)}{\log(1-w^s)}, \quad \text{rounded up}$$

### TODO 5: `ransac_iterations`

In [ ]:
def ransac_iterations(w, s, p=0.99):
    # w: inlier fraction, s: points per sample, p: wanted success probability. Returns an int.
    # TODO: the formula above, rounded up (math.ceil, math.log)
    ...

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def ransac_iterations(w, s, p=0.99):
    return math.ceil(math.log(1 - p) / math.log(1 - w ** s))

In [ ]:
# run this cell to check your code
tests = {(0.5, 2): 17, (0.6, 2): 11, (0.5, 4): 72, (0.3, 4): 567}
ok = all(ransac_iterations(w, s) == n for (w, s), n in tests.items())
print("toy line (w = 0.6, s = 2):", ransac_iterations(0.6, 2), "samples are enough; we used 100")
print("correct:", ok)

In [ ]:
ws = np.linspace(0.15, 0.95, 50)
plt.figure(figsize=(7, 4))
for s, name in [(2, "line (s = 2)"), (4, "homography (s = 4), next part")]:
    plt.plot(ws, [ransac_iterations(w, s) for w in ws], label=name)
plt.yscale("log")
plt.xlabel("inlier fraction w")
plt.ylabel("samples needed (p = 0.99)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Part 5: Homographies

A **homography** is a $3\times3$ matrix $H$ that says where each point of one photo lands in the other photo. It is
exact when the scene is flat, or when the camera only turns.

To move a point $(x, y)$ with $H$:

1. Add a 1: $(x, y, 1)$.
2. Multiply by $H$: you get $(x', y', w')$.
3. Divide by the last number: the new point is $(x'/w',\ y'/w')$.

$$\begin{bmatrix} x' \\ y' \\ w' \end{bmatrix} = H \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

Step 3 is what makes perspective possible (things far away get smaller). **4 matches** are enough to find $H$.

Simpler moves are homographies whose last row is $[0, 0, 1]$: then $w' = 1$ and step 3 changes nothing.

In [ ]:
grid = []
for t in np.linspace(0, 1, 11):
    for u in np.linspace(0, 1, 101):
        grid.append((t, u))
        grid.append((u, t))
grid = np.array(grid, np.float32)
c, s = np.cos(0.4), np.sin(0.4)
examples = {
    "translation": np.array([[1, 0, 0.3], [0, 1, 0.2], [0, 0, 1]]),
    "rotation + scale": np.array([[0.8 * c, -0.8 * s, 0.4], [0.8 * s, 0.8 * c, 0], [0, 0, 1]]),
    "affine": np.array([[1, 0.4, 0], [0.1, 0.7, 0.2], [0, 0, 1]]),
    "homography": np.array([[1, 0.1, 0], [0.1, 1, 0], [0.6, 0.3, 1]]),
}
fig, ax = plt.subplots(1, 4, figsize=(16, 4.5))
for a, (name, H_ex) in zip(ax, examples.items()):
    moved = cv2.perspectiveTransform(grid.reshape(-1, 1, 2), H_ex).reshape(-1, 2)
    a.plot(grid[:, 0], grid[:, 1], ".", ms=1, color="lightgray")
    a.plot(moved[:, 0], moved[:, 1], ".", ms=1, color="tab:blue")
    a.set_title(name + "\n" + np.array2string(H_ex, precision=2, suppress_small=True), fontsize=9)
    a.set_aspect("equal")
    a.set_xlim(-0.2, 1.6)
    a.set_ylim(1.4, -0.2)
plt.tight_layout()
plt.show()

Only the homography turns parallel lines into lines that meet, like a photo taken at an angle.

### TODO 6: `apply_homography`

In [ ]:
def apply_homography(H, pts):
    # H: (3, 3), pts: (N, 2) array of (x, y). Returns the mapped points, (N, 2).
    # TODO: add a column of ones -> (N, 3)
    # TODO: multiply every row by H  (rows @ H.T)
    # TODO: divide x' and y' by w'
    ...

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def apply_homography(H, pts):
    ones = np.ones((len(pts), 1))
    p = np.hstack([pts, ones])
    q = p @ H.T
    return q[:, :2] / q[:, 2:3]

In [ ]:
# run this cell to check your code
H_test = np.array([[0.9, 0.1, 30], [-0.2, 1.1, 10], [0.0005, 0.0002, 1]])
pts_test = np.random.default_rng(0).uniform(0, 500, (20, 2))
mine = apply_homography(H_test, pts_test)
theirs = cv2.perspectiveTransform(pts_test.reshape(-1, 1, 2), H_test).reshape(-1, 2)
print("correct:", mine.shape == (20, 2) and np.allclose(mine, theirs))

`cv2.findHomography(src, dst, cv2.RANSAC, 3.0)` finds $H$ from matched points:

- `src`: points in the first photo (here the zoomed photo),
- `dst`: the matching points in the second photo (the wide photo), in the same order.

It runs the RANSAC loop of Part 4, with 4 matches per try instead of 2 points:

1. pick 4 random matches and make $H$ from them,
2. move every `src` point with $H$,
3. a match is an inlier if it lands within 3 pixels of its `dst` point,
4. keep the $H$ with the most inliers.

It returns $H$ and `mask` (1 = inlier). Below we check the mask with our own `apply_homography`.

In [ ]:
H_zw, mask = cv2.findHomography(pts_zoom, pts_wide, cv2.RANSAC, 3.0)
inliers = mask.ravel() == 1
errors = np.linalg.norm(apply_homography(H_zw, pts_zoom) - pts_wide, axis=1)
print("H (zoom -> wide):")
print(np.round(H_zw, 4))
print(f"{inliers.sum()} inliers of {len(inliers)} matches")
print("inliers are exactly the matches with error < 3 pixels:", bool(np.all(inliers == (errors < 3))))

show_matches(zoom, wide, pts_zoom, pts_wide, ["lime" if k else "red" for k in inliers],
             "after RANSAC: green = inlier, red = outlier", lw=0.6)

Where was the zoomed photo taken? Map its 4 corners into the wide photo, and paste the zoomed photo there
(`cv2.warpPerspective` maps every pixel with $H$):

In [ ]:
h, w = zoom.shape[:2]
corners = np.array([[0, 0], [w, 0], [w, h], [0, h]], float)
outline = apply_homography(H_zw, corners)                     # step 1: the 4 corners, moved with H

warped = cv2.warpPerspective(zoom, H_zw, (wide.shape[1], wide.shape[0]))   # step 2: every pixel, moved with H
inside = warped.sum(axis=2) > 0                               # step 3: where the zoomed photo landed
pasted = wide.copy()
pasted[inside] = warped[inside]                               # step 4: paste it there

fig, ax = plt.subplots(2, 3, figsize=(16, 7.5))
ax[0, 0].imshow(zoom)
ax[0, 0].plot(corners[[0, 1, 2, 3, 0], 0], corners[[0, 1, 2, 3, 0], 1], "y", lw=4)
ax[0, 0].set_title("the zoomed photo and its 4 corners")
ax[0, 1].imshow(wide)
ax[0, 1].add_patch(plt.Polygon(outline, fill=False, color="yellow", lw=2))
ax[0, 1].set_title("step 1: the corners moved with H")
ax[0, 2].imshow(warped)
ax[0, 2].set_title("step 2: every pixel moved with H (warped)")
ax[1, 0].imshow(inside, cmap="gray")
ax[1, 0].set_title("step 3: where it landed (inside)")
ax[1, 1].imshow(pasted)
ax[1, 1].set_title("step 4: pasted onto the wide photo")
x0, y0 = (outline.min(axis=0) - 40).astype(int).clip(0)
x1, y1 = (outline.max(axis=0) + 40).astype(int)
ax[1, 2].imshow(pasted[y0:y1, x0:x1])
ax[1, 2].set_title("close-up: the sharper rock is from the zoomed photo")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

**A limit:** a homography is exact only for a flat scene or a camera that only turns. Here the camera moved closer,
so $H$ fits only roughly. The outliers are mostly in the water: ripples and reflections change between the two photos,
so their matches are wrong, and RANSAC throws them out.

## Part 6: Panorama

When the camera only **turns** (no moving), any two photos are related by a homography, whatever the scene looks like.
That is how a phone makes a panorama.

We simulate it: two views of the courtyard, as if the camera turned 11° left and 11° right (the right one a bit darker,
like a second shot). Since we made them, we know the true homography between them (`H_true`).

In [ ]:
def turn_camera(img, degrees, out_size=560, f=700.0):
    # the view of a camera turned by `degrees` around the vertical axis, and the homography that makes it
    t = np.deg2rad(degrees)
    R_turn = np.array([[np.cos(t), 0, -np.sin(t)], [0, 1, 0], [np.sin(t), 0, np.cos(t)]])
    K_in = np.array([[f, 0, img.shape[1] / 2], [0, f, img.shape[0] / 2], [0, 0, 1]])
    K_out = np.array([[f, 0, out_size / 2], [0, f, out_size / 2], [0, 0, 1]])
    H_turn = K_out @ R_turn @ np.linalg.inv(K_in)
    return cv2.warpPerspective(img, H_turn, (out_size, out_size)), H_turn


view_left, H_left = turn_camera(wide, -11)
view_right, H_right = turn_camera(wide, 11)
view_right = (view_right * 0.85).astype(np.uint8)    # a second shot is never exactly as bright as the first
H_true = H_left @ np.linalg.inv(H_right)          # right view -> left view
H_true = H_true / H_true[2, 2]

kp_l, desc_l = sift_features(view_left)
kp_r, desc_r = sift_features(view_right)
pairs_rl = match_ratio(desc_r, desc_l)
pts_r = np.array([kp_r[i].pt for i, j in pairs_rl])
pts_l = np.array([kp_l[j].pt for i, j in pairs_rl])
H_rl, mask_rl = cv2.findHomography(pts_r, pts_l, cv2.RANSAC, 3.0)
print(f"{len(pairs_rl)} matches, {mask_rl.sum()} inliers")
print("H found by RANSAC:")
print(np.round(H_rl / H_rl[2, 2], 4))
print("H_true:")
print(np.round(H_true, 4))

idx = np.nonzero(mask_rl.ravel())[0][::4]
show_matches(view_left, view_right, pts_l[idx], pts_r[idx], "lime", "left view / right view, inliers (every 4th)")

To stitch, map the right view into the left view's frame with $H$. The result does not fit in the left view's frame
(it sticks out on the right, and up and down), so first find the **canvas**: the box around the corners of the left view
and the mapped corners of the right view. If its top-left corner is $(x_{min}, y_{min})$, both views must be shifted by
$(-x_{min}, -y_{min})$, with the translation

$$T = \begin{bmatrix} 1 & 0 & -x_{min} \\ 0 & 1 & -y_{min} \\ 0 & 0 & 1 \end{bmatrix}$$

so the left view is warped with $T$ and the right view with $T H$.

### TODO 7: `stitch`

In [ ]:
def stitch(img1, img2, H):
    # H maps img2 points into img1. Returns the panorama (RGB uint8).
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    corners1 = np.array([[0, 0], [w1, 0], [w1, h1], [0, h1]], float)
    corners2 = np.array([[0, 0], [w2, 0], [w2, h2], [0, h2]], float)
    # TODO: map corners2 with H (apply_homography), stack them with corners1 -> 8 points
    # TODO: xmin, ymin = floor of the smallest x, y; xmax, ymax = ceil of the largest (as ints)
    # TODO: T = the translation matrix above; width = xmax - xmin, height = ymax - ymin
    T = ...
    width, height = ...
    warp1 = cv2.warpPerspective(img1, T, (width, height))
    warp2 = cv2.warpPerspective(img2, T @ H, (width, height))
    # given: average where both views have pixels, otherwise take the one that has them
    has1 = cv2.warpPerspective(np.ones((h1, w1), np.uint8), T, (width, height), flags=cv2.INTER_NEAREST) > 0
    has2 = cv2.warpPerspective(np.ones((h2, w2), np.uint8), T @ H, (width, height), flags=cv2.INTER_NEAREST) > 0
    both = has1 & has2
    pano = np.where(has1[:, :, None], warp1, warp2)
    pano[both] = (warp1[both].astype(np.uint16) + warp2[both]) // 2
    return pano

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def stitch(img1, img2, H):
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    corners1 = np.array([[0, 0], [w1, 0], [w1, h1], [0, h1]], float)
    corners2 = np.array([[0, 0], [w2, 0], [w2, h2], [0, h2]], float)
    all_corners = np.vstack([corners1, apply_homography(H, corners2)])
    xmin = int(np.floor(all_corners[:, 0].min()))
    ymin = int(np.floor(all_corners[:, 1].min()))
    xmax = int(np.ceil(all_corners[:, 0].max()))
    ymax = int(np.ceil(all_corners[:, 1].max()))
    T = np.array([[1, 0, -xmin], [0, 1, -ymin], [0, 0, 1]], float)
    width, height = xmax - xmin, ymax - ymin
    warp1 = cv2.warpPerspective(img1, T, (width, height))
    warp2 = cv2.warpPerspective(img2, T @ H, (width, height))
    has1 = cv2.warpPerspective(np.ones((h1, w1), np.uint8), T, (width, height), flags=cv2.INTER_NEAREST) > 0
    has2 = cv2.warpPerspective(np.ones((h2, w2), np.uint8), T @ H, (width, height), flags=cv2.INTER_NEAREST) > 0
    both = has1 & has2
    pano = np.where(has1[:, :, None], warp1, warp2)
    pano[both] = (warp1[both].astype(np.uint16) + warp2[both]) // 2
    return pano

In [ ]:
# run this cell to check your code
pano = stitch(view_left, view_right, H_rl)
c2 = apply_homography(H_true, np.array([[0, 0], [560, 0], [560, 560], [0, 560]], float))
true_w = np.ceil(max(560, c2[:, 0].max())) - np.floor(min(0, c2[:, 0].min()))
true_h = np.ceil(max(560, c2[:, 1].max())) - np.floor(min(0, c2[:, 1].min()))
print("panorama:", pano.shape, " expected about:", (int(true_h), int(true_w), 3))
ymin = int(np.floor(min(0, c2[:, 1].min())))
ok_left = np.abs(pano[-ymin + 280, 100].astype(int) - view_left[280, 100]).max() <= 2
print("correct:", bool(abs(pano.shape[0] - true_h) <= 3 and abs(pano.shape[1] - true_w) <= 3 and ok_left))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5), gridspec_kw={"width_ratios": [1, 1, pano.shape[1] / pano.shape[0]]})
ax[0].imshow(view_left)
ax[0].set_title("left view")
ax[1].imshow(view_right)
ax[1].set_title("right view")
ax[2].imshow(pano)
ax[2].set_title("panorama")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

The bow-tie shape is normal: the right view is a turned camera, so its rectangle becomes a trapezoid in the left view's
frame. You can see where the darker right view starts: phones also blend the seam smoothly and crop the black
borders.

## Part 7: Try it yourself

Take two photos with your phone: stand still and **turn** a little between them, so they overlap by about a third.
Save them as JPG (not HEIC), put them next to this notebook (on Colab: the folder icon on the left, then upload), and
write their names below. Without photos, the cell uses the two courtyard views.

If the result is broken: more overlap, less turning, and no moving between the shots (Part 5 says why).

In [ ]:
left_name = "my_left.jpg"
right_name = "my_right.jpg"


def shrink(img, longest=960):
    scale = longest / max(img.shape[:2])
    if scale >= 1:
        return img
    return cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)


if os.path.exists(left_name) and os.path.exists(right_name):
    img_l = shrink(load(left_name))
    img_r = shrink(load(right_name))
else:
    print("photos not found, using the courtyard views")
    img_l, img_r = view_left, view_right

kp1, desc1 = sift_features(img_l)
kp2, desc2 = sift_features(img_r)
pairs_try = match_ratio(desc2, desc1)
pts2 = np.array([kp2[i].pt for i, j in pairs_try])
pts1 = np.array([kp1[j].pt for i, j in pairs_try])
H_try, mask_try = cv2.findHomography(pts2, pts1, cv2.RANSAC, 3.0)
print(f"{len(pairs_try)} matches, {mask_try.sum()} inliers")

my_pano = stitch(img_l, img_r, H_try)
plt.figure(figsize=(14, 6))
plt.imshow(my_pano)
plt.axis("off")
plt.show()

## Extra: ORB, a fast binary descriptor

*In class if you have time, otherwise at home.*

SIFT's 128 floats are slow to compute and to compare. **ORB** finds corners with FAST (a quick test on a circle of
16 pixels around each point), gives them an angle, and describes each one with **256 bits**: is pixel $a$ brighter than
pixel $b$, for 256 fixed pairs $(a, b)$ in the patch. Two descriptors are compared with the **Hamming distance**: the
number of bits that differ. OpenCV packs the 256 bits into 32 bytes (`uint8`).

### TODO 8: `hamming`

In [ ]:
def hamming(a, b):
    # a, b: uint8 arrays of 32 bytes (256 bits). Returns the number of bits that differ (an int).
    # TODO: XOR the bytes (np.bitwise_xor): a 1 bit where they differ
    # TODO: count the 1 bits: np.unpackbits(...) turns bytes into bits
    ...

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def hamming(a, b):
    return int(np.unpackbits(np.bitwise_xor(a, b)).sum())

In [ ]:
# run this cell to check your code
orb = cv2.ORB_create(nfeatures=2000)
kp_o, desc_o = orb.detectAndCompute(cv2.cvtColor(wide, cv2.COLOR_RGB2GRAY), None)
print("ORB descriptors:", desc_o.shape, desc_o.dtype)
ok = all(hamming(desc_o[i], desc_o[i + 1]) == int(cv2.norm(desc_o[i], desc_o[i + 1], cv2.NORM_HAMMING)) for i in range(50))
print("correct:", ok)

SIFT against ORB on the real pair (zoom → wide): time, matches and RANSAC inliers. For ORB the matcher uses the
Hamming distance, `cv2.NORM_HAMMING`.

In [ ]:
gray_zoom = cv2.cvtColor(zoom, cv2.COLOR_RGB2GRAY)
gray_wide = cv2.cvtColor(wide, cv2.COLOR_RGB2GRAY)
methods = {"SIFT": (cv2.SIFT_create(nfeatures=2000), cv2.NORM_L2),
           "ORB": (cv2.ORB_create(nfeatures=2000), cv2.NORM_HAMMING)}
results = {}
for name, (det, norm) in methods.items():
    t0 = time.time()
    for repeat in range(5):
        k1, d1 = det.detectAndCompute(gray_zoom, None)
        k2, d2 = det.detectAndCompute(gray_wide, None)
    t_detect = (time.time() - t0) / 5
    knn = cv2.BFMatcher(norm).knnMatch(d1, d2, k=2)
    good = [m for m, n in knn if m.distance < 0.75 * n.distance]
    p1 = np.array([k1[m.queryIdx].pt for m in good])
    p2 = np.array([k2[m.trainIdx].pt for m in good])
    H_m, mask_m = cv2.findHomography(p1, p2, cv2.RANSAC, 3.0)
    results[name] = (p1, p2, mask_m.ravel() == 1)
    print(f"{name:4}: {1000 * t_detect:4.0f} ms for both photos, {len(good)} matches, {mask_m.sum()} inliers, "
          f"descriptor = {d1.shape[1]} x {d1.dtype}")

p1, p2, inl_orb = results["ORB"]
show_matches(zoom, wide, p1[inl_orb], p2[inl_orb], "cyan", "ORB, RANSAC inliers", lw=0.6)

A SIFT descriptor takes 512 bytes (128 floats), an ORB descriptor 32 bytes, and comparing bits is much cheaper than
comparing floats. That is why ORB is used when speed and memory matter (phones, robots, real-time tracking, ORB-SLAM),
and SIFT when we want the most reliable matches.